# CTX - Context Analyzer

CTX is a diagnostic tool for analyzing LLM conversation token usage and detecting when conversations enter the "dumb zone" - a region of the context window where model performance may degrade.

## Key Concepts

- **Dumb Zone**: The portion of the context window (default: after 40% utilization) where model performance can degrade
- **Token Categories**: Tokens are categorized as `system`, `user`, `assistant`, or `tools`
- **Round Tracking**: Each message is tracked as a "round" with cumulative token counts


In [1]:
from rich import print as rprint

## Basic Usage

### Quick Status Check

The fastest way to check if a conversation is approaching the dumb zone:


In [ ]:
from tinyloop.ctx import get_status

# Sample conversation
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant specializing in Python programming.",
    },
    {"role": "user", "content": "How do I read a file in Python?"},
    {
        "role": "assistant",
        "content": "You can read a file in Python using the built-in `open()` function. Here's a simple example:\n\n```python\nwith open('filename.txt', 'r') as file:\n    content = file.read()\n    print(content)\n```\n\nThe `with` statement ensures the file is properly closed after reading.",
    },
    {"role": "user", "content": "What about reading line by line?"},
    {
        "role": "assistant",
        "content": "To read a file line by line, you can iterate over the file object:\n\n```python\nwith open('filename.txt', 'r') as file:\n    for line in file:\n        print(line.strip())\n```\n\nOr use `readlines()` to get a list of all lines:\n\n```python\nwith open('filename.txt', 'r') as file:\n    lines = file.readlines()\n```",
    },
]

# Get quick status
status = get_status(messages)
print(status.message)
print(f"\nTotal tokens: {status.total_tokens}")
print(f"Threshold: {status.threshold_tokens}")
print(f"In dumb zone: {status.is_in_dumb_zone}")
print(f"Tokens until dumb zone: {status.tokens_until_dumb_zone}")

  192 / 67,200 tokens (0.1%) -- 67,008 tokens until dumb zone

Total tokens: 192
Threshold: 67200
In dumb zone: False
Tokens until dumb zone: 67008


### Full Analysis with CTXAnalyzer

For detailed analysis including round-by-round breakdown:


In [3]:
from tinyloop.ctx import CTXAnalyzer

# Create analyzer with custom settings
analyzer = CTXAnalyzer(
    model="anthropic/claude-sonnet-4-20250514",
    context_window=168000,  # 168k tokens
    threshold=0.4,  # 40% = dumb zone
    offline=True,  # Use tiktoken instead of API for this demo
)

# Analyze conversation
result = analyzer.analyze(messages)

rprint(result)

CTXResult(
    model='anthropic/claude-sonnet-4-20250514',
    context_window=168000,
    threshold=0.4,
    threshold_tokens=67200,
    total_tokens=189,
    percentage_used=0.001125,
    is_in_dumb_zone=False,
    dumb_zone_round=None,
    categories={'system': 14, 'user': 24, 'assistant': 151, 'tools': 0},
    categories_detail=TokenCategories(
        system=14,
        user=24,
        assistant=151,
        tools=0,
        tools_breakdown=CategoryBreakdown(
            total=0,
            tool_definitions=0,
            tool_calls=0,
            tool_responses=0,
            text_content=0,
            images=0
        ),
        user_breakdown=CategoryBreakdown(
            total=24,
            tool_definitions=0,
            tool_calls=0,
            tool_responses=0,
            text_content=24,
            images=0
        )
    ),
    rounds=[
        RoundResult(
            round_number=1,
            role='system',
            tokens=14,
            cumulative_tokens=14,
            percentage_used=8.333333333333333e-05,
            is_in_dumb_zone=False
        ),
        RoundResult(
            round_number=2,
            role='user',
            tokens=13,
            cumulative_tokens=27,
            percentage_used=0.0001607142857142857,
            is_in_dumb_zone=False
        ),
        RoundResult(
            round_number=3,
            role='assistant',
            tokens=67,
            cumulative_tokens=94,
            percentage_used=0.0005595238095238096,
            is_in_dumb_zone=False
        ),
        RoundResult(
            round_number=4,
            role='user',
            tokens=11,
            cumulative_tokens=105,
            percentage_used=0.000625,
            is_in_dumb_zone=False
        ),
        RoundResult(
            round_number=5,
            role='assistant',
            tokens=84,
            cumulative_tokens=189,
            percentage_used=0.001125,
            is_in_dumb_zone=False
        )
    ],
    tokenizer_used='tiktoken (cl100k_base)'
)

### Inspecting Analysis Results


In [4]:
# Summary
print(f"Model: {result.model}")
print(f"Context Window: {result.context_window:,} tokens")
print(f"Threshold: {result.threshold * 100}% ({result.threshold_tokens:,} tokens)")
print(f"Tokenizer: {result.tokenizer_used}")
print()
print(f"Total Tokens: {result.total_tokens:,}")
print(f"Percentage Used: {result.percentage_used * 100:.2f}%")
print(f"In Dumb Zone: {result.is_in_dumb_zone}")

Model: anthropic/claude-sonnet-4-20250514
Context Window: 168,000 tokens
Threshold: 40.0% (67,200 tokens)
Tokenizer: tiktoken (cl100k_base)

Total Tokens: 189
Percentage Used: 0.11%
In Dumb Zone: False


In [5]:
# Token categories
print("Token Distribution:")
for category, tokens in result.categories.items():
    pct = tokens / result.total_tokens * 100 if result.total_tokens > 0 else 0
    print(f"  {category}: {tokens:,} ({pct:.1f}%)")

Token Distribution:
  system: 14 (7.4%)
  user: 24 (12.7%)
  assistant: 151 (79.9%)
  tools: 0 (0.0%)


In [ ]:
# Round-by-round breakdown
print("Round-by-Round Analysis:")
print(
    f"{'Round':<6} {'Role':<12} {'Tokens':<10} {'Cumulative':<12} {'% Used':<10} {'Status'}"
)
print("-" * 65)
for r in result.rounds:
    status = "DUMB ZONE" if r.is_in_dumb_zone else "Safe"
    print(
        f"{r.round_number:<6} {r.role:<12} {r.tokens:<10} {r.cumulative_tokens:<12} {r.percentage_used * 100:<10.2f} {status}"
    )

Round-by-Round Analysis:
Round  Role         Tokens     Cumulative   % Used     Status
-----------------------------------------------------------------
1      system       14         14           0.01       Safe
2      user         13         27           0.02       Safe
3      assistant    67         94           0.06       Safe
4      user         11         105          0.06       Safe
5      assistant    84         189          0.11       Safe


## Detecting Dumb Zone Entry

Let's create a longer conversation that will enter the dumb zone:


In [ ]:
# Create a conversation that will exceed threshold
# Using a small context window for demonstration
long_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
]

# Add many messages to fill context
for i in range(20):
    long_messages.append(
        {"role": "user", "content": f"This is user message {i}. " * 20}
    )
    long_messages.append(
        {"role": "assistant", "content": f"This is assistant response {i}. " * 30}
    )

# Analyze with a smaller context window to trigger dumb zone
small_analyzer = CTXAnalyzer(
    context_window=5000,  # Small context window for demo
    threshold=0.4,
    offline=True,
)

result = small_analyzer.analyze(long_messages)

print(f"Total tokens: {result.total_tokens:,}")
print(f"Threshold: {result.threshold_tokens:,}")
print(f"In dumb zone: {result.is_in_dumb_zone}")
print(f"Dumb zone entered at round: {result.dumb_zone_round}")

Total tokens: 7,210
Threshold: 2,000
In dumb zone: True
Dumb zone entered at round: 13


## Analyzing Conversations with Tools

CTX can also analyze conversations that include tool definitions and tool calls:


In [ ]:
# Conversation with tool calls
tool_messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant with access to weather data.",
    },
    {"role": "user", "content": "What's the weather in San Francisco?"},
    {
        "role": "assistant",
        "content": "Let me check the weather for you.",
        "tool_calls": [
            {
                "id": "call_123",
                "type": "function",
                "function": {
                    "name": "get_weather",
                    "arguments": '{"location": "San Francisco, CA"}',
                },
            }
        ],
    },
    {
        "role": "tool",
        "tool_call_id": "call_123",
        "content": '{"temperature": 68, "condition": "sunny", "humidity": 45}',
    },
    {
        "role": "assistant",
        "content": "The weather in San Francisco is currently sunny with a temperature of 68°F and 45% humidity.",
    },
]

# Tool definitions
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"],
            },
        },
    }
]

# Analyze with tools
result = analyzer.analyze(tool_messages, tools=tools)

print("Token Distribution with Tools:")
for category, tokens in result.categories.items():
    print(f"  {category}: {tokens:,}")

print(f"\nTools category includes: tool definitions, tool calls, and tool responses")

Token Distribution with Tools:
  system: 15
  user: 12
  assistant: 25
  tools: 133

Tools category includes: tool definitions, tool calls, and tool responses


## LLM Integration with Middleware

CTX provides middleware to monitor context usage during live LLM conversations:


In [9]:
from tinyloop.ctx import CTXMiddleware

# Create middleware
ctx = CTXMiddleware(
    context_window=168000,
    threshold=0.4,
    action="warn",  # Print warning when threshold exceeded
    offline=True,
)

# Check status of our sample messages
status = ctx.check(messages=messages)
print(f"Status message: {status.message}")
print(f"Is approaching threshold: {status.is_approaching}")

Status message:   192 / 67,200 tokens (0.1%) -- 67,008 tokens until dumb zone
Is approaching threshold: False


In [10]:
# Full analysis via middleware
result = ctx.analyze(messages=messages)
print(f"Total tokens: {result.total_tokens}")
print(f"Categories: {result.categories}")

Total tokens: 189
Categories: {'system': 14, 'user': 24, 'assistant': 151, 'tools': 0}


### Using with Real LLM Instance


In [ ]:
from tinyloop import LLM
from tinyloop.ctx import CTXMiddleware

# Create LLM and middleware
llm = LLM(
    model="openai/gpt-3.5-turbo",
    temperature=0.1,
    system_prompt="You are a helpful coding assistant.",
)

ctx = CTXMiddleware(
    context_window=16000,  # GPT-3.5-turbo context window
    threshold=0.4,
    action="warn",
)

# Have a conversation
response = llm(prompt="What is a Python list comprehension?")
print("Response received.")

# Check context status
status = ctx.check(llm)
print(f"\nContext status: {status.message}")

Response received.

Context status:   99 / 6,400 tokens (0.6%) -- 6,301 tokens until dumb zone


In [12]:
# Continue conversation
response = llm(prompt="Can you give me 5 examples?")
print("Response received.")

# Check status again
status = ctx.check(llm)
print(f"\nContext status: {status.message}")
print(f"Message history length: {len(llm.get_history())}")

Response received.

Context status:   543 / 6,400 tokens (3.4%) -- 5,857 tokens until dumb zone
Message history length: 5


### Raise Exception on Threshold


In [ ]:
from tinyloop.ctx import CTXMiddleware, CTXThresholdExceeded

# Create strict middleware that raises on threshold
ctx_strict = CTXMiddleware(
    context_window=100,  # Very small for demo
    threshold=0.1,  # 10 tokens threshold
    action="raise",
)

# This will raise an exception
try:
    ctx_strict.check(messages=messages)
except CTXThresholdExceeded as e:
    print(f"Exception caught!")
    print(f"Total tokens: {e.total_tokens}")
    print(f"Threshold tokens: {e.threshold_tokens}")
    print(f"Percentage used: {e.percentage_used:.1%}")

### Custom Warning Callback


In [ ]:
from tinyloop.ctx import CTXMiddleware


# Custom warning handler
def my_warning_handler(status):
    print("=" * 50)
    print("CUSTOM WARNING: Context threshold exceeded!")
    print(f"Tokens: {status.total_tokens:,} / {status.threshold_tokens:,}")
    print(f"Percentage: {status.percentage_used:.1%}")
    print("Consider summarizing the conversation.")
    print("=" * 50)


# Create middleware with custom callback
ctx_custom = CTXMiddleware(
    context_window=100,  # Small for demo
    threshold=0.1,
    action="warn",
    on_warning=my_warning_handler,
)

# This will trigger the custom warning
ctx_custom.check(messages=messages)

## Simple Hook Function

For simple use cases, you can create a hook function:


In [13]:
from tinyloop.ctx import create_ctx_hook

# Create a simple hook
check_ctx = create_ctx_hook(
    context_window=168000,
    threshold=0.4,
    action="warn",
)

# Use it to check any messages
status = check_ctx(messages=messages)
print(f"Quick check: {status.message}")

Quick check:   192 / 67,200 tokens (0.1%) -- 67,008 tokens until dumb zone


## CLI Usage

CTX also provides a command-line interface. Here are some examples:

```bash
# Full report with TUI tables
tinyloop ctx conversation.json

# Simple one-line status
tinyloop ctx -s conversation.json

# Pipe from another command
cat conversation.json | tinyloop ctx

# Custom threshold (30%) and context window (200k)
tinyloop ctx -t 0.3 -c 200000 conversation.json

# Specify model for accurate tokenization
tinyloop ctx -m anthropic/claude-sonnet-4-20250514 conversation.json

# Verbose output with detailed breakdowns
tinyloop ctx -v conversation.json

# Offline mode (no API calls)
tinyloop ctx --offline conversation.json
```

### Input Format

The CLI accepts JSON files in these formats:

**Full format:**

```json
{
  "model": "anthropic/claude-sonnet-4-20250514",
  "context_window": 200000,
  "messages": [...],
  "tools": [...]
}
```

**Minimal format:**

```json
{
  "messages": [...]
}
```

**Direct array:**

```json
[
  { "role": "user", "content": "Hello" },
  { "role": "assistant", "content": "Hi!" }
]
```

### Exit Codes

- `0`: Success, not in dumb zone
- `1`: Success, in dumb zone (useful for CI/CD)
- `2`: Input error
- `3`: Tokenizer error


## Save Conversation for CLI Testing


In [ ]:
import json

# Save our sample conversation
conversation_data = {
    "model": "anthropic/claude-sonnet-4-20250514",
    "messages": messages,
}

with open("sample_conversation.json", "w") as f:
    json.dump(conversation_data, f, indent=2)

print("Saved to sample_conversation.json")
print("\nTry running:")
print("  tinyloop ctx sample_conversation.json")
print("  tinyloop ctx -s sample_conversation.json")

## Best Practices

1. **Monitor Regularly**: Check context status after each LLM call in long conversations

2. **Set Appropriate Thresholds**: The default 40% threshold is conservative. Adjust based on your use case.

3. **Use Offline Mode for Development**: When developing, use `offline=True` to avoid API calls for token counting.

4. **Handle Dumb Zone Gracefully**: When the dumb zone is reached, consider:

   - Summarizing the conversation
   - Pruning older messages
   - Starting a new conversation with context

5. **Track Tool Usage**: Tool definitions and responses can consume significant tokens. Monitor the `tools` category.
